# 第10课 - 生产环境中的 AI 代理

在本课中，您将学习使用 Microsoft Agent Framework（Python）构建 AI 代理的**生产模式**。我们将涵盖：

- **可观测性** — 为代理交互添加计时和日志记录
- **评估** — 使用评估器代理对响应质量进行评分
- **成本管理** — 令牌优化和模型选择的策略

该场景是一个**旅游代理**，帮助用户规划行程，并在其上叠加监控和评估。


## 设置


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## 生产注意事项

将 AI 代理从原型推向生产需要仔细关注三大支柱：

1. **可观测性** — 你需要了解代理在做什么、执行了多长时间以及调用了哪些工具。没有追踪和日志记录，调试生产问题几乎不可能。

2. **评估** — 自动化质量检查可确保代理的响应随时间保持准确、完整且有用。评估代理可以根据定义的标准为响应评分。

3. **成本管理** — 令牌使用量会直接影响成本。像提示词优化、模型选择和缓存这样的策略有助于在不牺牲质量的情况下控制开支。


## 构建可观测的代理

我们定义了旅行工具，并对代理调用进行计时包装，以便我们可以监控延迟。在生产环境中，您会将其与 OpenTelemetry 或类似的追踪后端集成。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 评估模式

一种常见的生产模式是使用第二个代理作为**评估者**。评估者根据预定义的标准（例如完整性、准确性和有用性）对主要代理的响应进行评分。

这可以实现：
- 在响应到达用户之前的自动化质量关卡
- 在提示或模型更改时的回归检测
- 持续监控代理性能随时间的变化


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 成本管理策略

控制成本对生产环境中的 AI 代理至关重要。以下是关键策略：

| 策略 | 说明 |
|---|---|
| **提示优化** | 保持系统指令简洁。移除冗余上下文以减少输入令牌。 |
| **模型选择** | 使用更小、更便宜的模型 (例如 GPT-4o-mini) 来处理诸如分类或抽取这样的简单任务，并将更大的模型保留给复杂推理。 |
| **缓存** | 缓存工具结果和常见查询以避免重复的 API 调用。 |
| **令牌预算** | 设置 `max_tokens` 限制以防止响应意外过长。 |
| **批处理** | 在可能的情况下，将多个用户查询合并到一次 API 调用中。 |

在实践中，分层方法效果很好：将简单的请求路由到快速且廉价的模型，只将复杂的查询升级到更强大的模型。


## 总结

在本课中，您学会了如何：

1. **为代理交互添加可观测性**，通过计时和日志记录，为追踪和监控奠定基础。
2. **自动评估代理响应**，使用评估器代理对完整性、准确性和有用性进行评分。
3. **管理成本**，通过提示优化、模型选择、缓存和令牌预算。

这些生产模式有助于确保您的 AI 代理在大规模部署时可靠、可度量且具有成本效益。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免责声明：
本文件已使用 AI 翻译服务 Co-op Translator（https://github.com/Azure/co-op-translator）进行翻译。尽管我们力求准确，但请注意自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议采用专业人工翻译。我们不对因使用此翻译而产生的任何误解或错误解释承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
